# Objective

The objective of this project is to build a Machine Learning regression model capable of predicting the total delivery time of an order in minutes.


# Step 1 — Data Loading and Cleaning

The objective of this step is to understand the structure of the dataset and transform incorrect or inconsistent raw values into usable data.

Only deterministic cleaning operations are performed at this stage.

Missing-value imputation using the median, mode, etc. will be performed later inside a scikit-learn Pipeline to avoid data leakage.

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [56]:
df = pd.read_csv(
    "./../data/food-delivery.csv"
)

In [ ]:
df_raw = df.copy()

<ArrowStringArray>
['Urban', 'Metropolitian', 'Semi-Urban', nan]
Length: 4, dtype: str


## 1.1 Dataset dimensions

In [57]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 45593
Columns: 20


## 1.2 Dataset preview

In [11]:
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## 1.3 General information

In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle_condition          

In [60]:
df.columns

Index(['ID', 'Delivery_person_ID', 'Delivery_person_Age',
       'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date', 'Time_Orderd',
       'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density',
       'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle',
       'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)'],
      dtype='str')

## 2. Column roles

| Group | Columns |
|---|---|
| Identifiers | `ID`, `Delivery_person_ID` |
| Courier information | `Delivery_person_Age`, `Delivery_person_Ratings` |
| GPS coordinates | Restaurant and delivery latitude/longitude |
| Date/time | `Order_Date`, `Time_Orderd`, `Time_Order_picked` |
| Environment | `Weatherconditions`, `Road_traffic_density` |
| Delivery information | `Vehicle_condition`, `Type_of_order`, `Type_of_vehicle`, `multiple_deliveries`, `Festival`, `City` |
| Target | `Time_taken(min)` |

`ID` and `Delivery_person_ID` both are unique order identifier and will not be used as a predictive feature.

In [61]:
df.isnull().sum()

ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64

## 3. Detect text-encoded missing values

Some missing values are represented by the string `"NaN"` instead of a real null value.

These values must first be converted to `np.nan`.

In [62]:
string_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in string_columns:
    df[column] = df[column].str.strip()

In [66]:
df = df.replace("NaN", np.nan)

In [67]:
df["Weatherconditions"] = (
    df["Weatherconditions"]
    .str.replace("conditions ", "", regex=False)
)

In [68]:
df["Weatherconditions"].value_counts(
    dropna=False
)

Weatherconditions
Fog           7654
Stormy        7586
Cloudy        7536
Sandstorms    7495
Windy         7422
Sunny         7284
NaN            616
Name: count, dtype: int64

## 4. Clean the target variable

`Time_taken(min)` contains textual information such as `(min) 24`.

The numerical delivery duration will be extracted and converted to a numeric variable.

In [69]:
df["Time_taken_min"] = pd.to_numeric(
    df["Time_taken(min)"]
    .str.extract(r"(\d+)", expand=False),
    errors="coerce"
)

In [70]:
df = df.drop(
    columns=["Time_taken(min)"]
)

In [71]:
print(df["Time_taken_min"].head())
df["Time_taken_min"].dtype

0    24
1    33
2    26
3    21
4    30
Name: Time_taken_min, dtype: int64


dtype('int64')

In [73]:
numeric_to_convert = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "multiple_deliveries"
]

for column in numeric_to_convert:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

In [74]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [75]:
df["Order_Date"].dtype

dtype('<M8[us]')

In [76]:
missing_data = pd.DataFrame({
    "Missing_values": df.isnull().sum(),
    "Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_data

,Missing_values,Percentage
ID,0,0.00
Delivery_person_ID,0,0.00
Delivery_person_Age,1854,4.07
Delivery_person_Ratings,1908,4.18
Restaurant_latitude,0,0.00
Restaurant_longitude,0,0.00
Delivery_location_latitude,0,0.00
Delivery_location_longitude,0,0.00
Order_Date,0,0.00
Time_Orderd,1731,3.80


In [77]:
df_raw.isnull().sum()

ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64

In [79]:
categorical_columns = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(
        df[column].value_counts(
            dropna=False
        )
    )


--- Weatherconditions ---
Weatherconditions
Fog           7654
Stormy        7586
Cloudy        7536
Sandstorms    7495
Windy         7422
Sunny         7284
NaN            616
Name: count, dtype: int64

--- Road_traffic_density ---
Road_traffic_density
Low       15477
Jam       14143
Medium    10947
High       4425
NaN         601
Name: count, dtype: int64

--- Type_of_order ---
Type_of_order
Snack     11533
Meal      11458
Drinks    11322
Buffet    11280
Name: count, dtype: int64

--- Type_of_vehicle ---
Type_of_vehicle
motorcycle          26435
scooter             15276
electric_scooter     3814
bicycle                68
Name: count, dtype: int64

--- Festival ---
Festival
No     44469
Yes      896
NaN      228
Name: count, dtype: int64

--- City ---
City
Metropolitian    34093
Urban            10136
NaN               1200
Semi-Urban         164
Name: count, dtype: int64


In [80]:
print(
    df["Delivery_person_Age"]
    .value_counts()
    .sort_index()
)

Delivery_person_Age
15.0      38
20.0    2136
21.0    2153
22.0    2196
23.0    2087
24.0    2210
25.0    2174
26.0    2159
27.0    2150
28.0    2179
29.0    2191
30.0    2226
31.0    2120
32.0    2202
33.0    2187
34.0    2166
35.0    2262
36.0    2260
37.0    2227
38.0    2219
39.0    2144
50.0      53
Name: count, dtype: int64


In [81]:
print(
    df["Delivery_person_Ratings"]
    .value_counts()
    .sort_index()
)

Delivery_person_Ratings
1.0      38
2.5      20
2.6      22
2.7      22
2.8      19
2.9      19
3.0       6
3.1      29
3.2      29
3.3      25
3.4      32
3.5     249
3.6     207
3.7     225
3.8     228
3.9     197
4.0    1077
4.1    1430
4.2    1418
4.3    1409
4.4    1361
4.5    3303
4.6    6940
4.7    7142
4.8    7148
4.9    7041
5.0    3996
6.0      53
Name: count, dtype: int64


### Courier anomaly decision

The values `Age = 15 / Rating = 1` and `Age = 50 / Rating = 6` form a systematic anomalous pattern.

They represent only a very small proportion of the dataset and do not follow the normal distribution of courier information.

Instead of deleting the entire observations, the suspicious age and rating values will be converted to missing values.

They can later be imputed inside the machine-learning preprocessing pipeline.

In [83]:
courier_anomaly = (
    (
        (df["Delivery_person_Age"] == 15) &
        (df["Delivery_person_Ratings"] == 1)
    )
    |
    (
        (df["Delivery_person_Age"] == 50) &
        (df["Delivery_person_Ratings"] == 6)
    )
)

print(
    "Suspicious courier rows:",
    courier_anomaly.sum()
)

Suspicious courier rows: 91


In [84]:
df.loc[
    courier_anomaly,
    [
        "Delivery_person_Age",
        "Delivery_person_Ratings"
    ]
] = np.nan

In [85]:
coordinate_columns = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

df[coordinate_columns].describe()

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
count,45593.000000,45593.000000,45593.000000,45593.000000
mean,17.017729,70.231332,17.465186,70.845702
std,8.185109,22.883647,7.335122,21.118812
min,-30.905562,-88.366217,0.010000,0.010000
25%,12.933284,73.170000,12.988453,73.280000
50%,18.546947,75.898497,18.633934,76.002574
75%,22.728163,78.044095,22.785049,78.107044
max,30.914057,88.433452,31.054057,88.563452


In [86]:
for column in coordinate_columns:
    print(
        column,
        "→ zero values:",
        (df[column] == 0).sum()
    )

Restaurant_latitude → zero values: 3640
Restaurant_longitude → zero values: 3640
Delivery_location_latitude → zero values: 0
Delivery_location_longitude → zero values: 0


In [87]:
print(
    "Negative restaurant latitudes:",
    (df["Restaurant_latitude"] < 0).sum()
)

print(
    "Negative restaurant longitudes:",
    (df["Restaurant_longitude"] < 0).sum()
)

Negative restaurant latitudes: 431
Negative restaurant longitudes: 162


In [88]:
df.loc[
    (df["Restaurant_latitude"] < 0) |
    (df["Restaurant_longitude"] < 0),
    coordinate_columns
].head()

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
92,-27.163303,78.057044,27.233303,78.127044
283,-27.165108,78.015053,27.225108,78.075053
289,-18.551440,-73.804855,18.611440,73.864855
425,-30.366322,-78.070453,30.496322,78.200453
534,-12.325461,-76.632278,12.385461,76.692278


In [45]:
df["Restaurant_latitude"] = (
    df["Restaurant_latitude"].abs()
)

df["Restaurant_longitude"] = (
    df["Restaurant_longitude"].abs()
)

In [89]:
invalid_coordinates = (
    (df["Restaurant_latitude"] == 0) |
    (df["Restaurant_longitude"] == 0)
)

print(
    "Rows with invalid GPS coordinates:",
    invalid_coordinates.sum()
)

Rows with invalid GPS coordinates: 3640


In [47]:
df.loc[
    invalid_coordinates,
    coordinate_columns
] = np.nan

In [48]:
print(
    df["Time_Orderd"]
    .value_counts(dropna=False)
    .head()
)

print()

print(
    df["Time_Order_picked"]
    .value_counts(dropna=False)
    .head()
)

Time_Orderd
NaN         1731
21:55:00     461
17:55:00     456
20:00:00     449
22:20:00     448
Name: count, dtype: int64

Time_Order_picked
21:30:00    496
22:50:00    474
22:40:00    458
18:40:00    457
21:45:00    456
Name: count, dtype: int64


In [49]:
print(
    "Duplicated rows:",
    df.duplicated().sum()
)

print(
    "Duplicated ID:",
    df["ID"].duplicated().sum()
)

Duplicated rows: 0
Duplicated ID: 0


In [50]:
df["Time_taken_min"].describe()

count    45593.000000
mean        26.294607
std          9.383806
min         10.000000
25%         19.000000
50%         26.000000
75%         32.000000
max         54.000000
Name: Time_taken_min, dtype: float64

In [91]:
Q1 = df["Time_taken_min"].quantile(0.25)
Q3 = df["Time_taken_min"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

target_outliers = df[
    (df["Time_taken_min"] < lower_bound) |
    (df["Time_taken_min"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(target_outliers))

Lower bound: -0.5
Upper bound: 51.5
Number of outliers: 270


### Target outlier decision

The IQR method identifies some delivery durations above approximately 51.5 minutes as statistical outliers.

However, the maximum observed duration is only 54 minutes.

These values are realistic delayed deliveries rather than impossible observations.

They are therefore retained, particularly because detecting long delivery situations is part of the business objective.

In [52]:
missing_data = pd.DataFrame({
    "Missing_values": df.isnull().sum(),
    "Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_data[
    missing_data["Missing_values"] > 0
].sort_values(
    "Missing_values",
    ascending=False
)

,Missing_values,Percentage
Restaurant_longitude,3640,7.98
Restaurant_latitude,3640,7.98
Delivery_location_longitude,3640,7.98
Delivery_location_latitude,3640,7.98
Delivery_person_Ratings,1999,4.38
Delivery_person_Age,1945,4.27
Time_Orderd,1731,3.80
City,1200,2.63
multiple_deliveries,993,2.18
Weatherconditions,616,1.35


## Step 1 — Final decisions

| Variable | Decision |
|---|---|
| `ID` | Keep for traceability, exclude from training |
| `Delivery_person_ID` | Keep for traceability, probably exclude as a high-cardinality identifier |
| `Delivery_person_Age` | Convert to numeric; suspicious courier records converted to missing |
| `Delivery_person_Ratings` | Convert to numeric; suspicious courier records converted to missing |
| GPS coordinates | Correct negative restaurant signs; zero/dummy coordinates converted to missing |
| `Order_Date` | Convert to datetime |
| `Time_Orderd` | Keep; missing values handled later |
| `Time_Order_picked` | Keep |
| `Weatherconditions` | Remove `conditions` text and convert textual NaN to missing |
| `Road_traffic_density` | Clean categorical variable; missing values handled later |
| `Vehicle_condition` | Keep and investigate as a coded feature |
| `Type_of_order` | Keep |
| `Type_of_vehicle` | Keep |
| `multiple_deliveries` | Convert to numeric; missing values handled later |
| `Festival` | Keep; missing values handled later |
| `City` | Keep; missing values handled later |
| `Time_taken_min` | Extract numerical target; realistic high values retained |

## Step 1 conclusion

The dataset contains 45,593 delivery observations and 20 original variables.

Several data-quality issues were identified:

- numerical values stored as text;
- missing values encoded as strings;
- unnecessary text in weather conditions;
- textual information embedded in the target variable;
- invalid or suspicious courier information;
- negative restaurant coordinate signs;
- zero/dummy GPS coordinates;
- missing categorical and temporal information.

Deterministic formatting and validity problems were corrected without unnecessarily deleting complete observations.

Missing values were intentionally retained as `NaN` after cleaning. Their imputation will be learned only from the training data through a scikit-learn Pipeline, preventing data leakage.

The cleaned dataset is now suitable for exploratory data analysis and feature engineering.

In [94]:
print(df["Delivery_person_Ratings"].isna().sum())

1999
